In [34]:
from Code.ingest import load_faq_data
documents = load_faq_data()

In [35]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [36]:
documents = documents_llm

In [37]:
doc = documents[0]

In [38]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [39]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [40]:
import json

user_prompt = json.dumps(doc)

In [41]:
from dotenv import load_dotenv
import os 
from openai import OpenAI

load_dotenv()
or_client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
)


In [42]:
messages = [
    {"role" : "developer" , "content" : data_gen_instructions},
    {"role" : "user", "content" : user_prompt}
]

In [43]:
response = or_client.responses.parse(
    model="tencent/hy3:free",
    input=messages,
    text_format=Questions
)

In [44]:
result = response.output_parsed

print(result)
print(result.questions)

questions=['Late enrollee here—does handing last assignment before window closes earn finish credential?', 'Yo, jumped into this program behind schedule; turning in final work ahead of deadline get me proof of completion?', 'Someone starting after kickoff—will turning in end task before entries close secure certification?', 'Newbie hopped on board late; eligible for wrap-up badge provided final piece dropped before cutoff?', 'Missed early days of bootcamp—does handing capstone ahead of closure get completion proof?']
['Late enrollee here—does handing last assignment before window closes earn finish credential?', 'Yo, jumped into this program behind schedule; turning in final work ahead of deadline get me proof of completion?', 'Someone starting after kickoff—will turning in end task before entries close secure certification?', 'Newbie hopped on board late; eligible for wrap-up badge provided final piece dropped before cutoff?', 'Missed early days of bootcamp—does handing capstone ahead

In [45]:
import Code.rag_helper

In [46]:
from Code.evaluation_utils import llm_structured

In [47]:
result, usage = llm_structured(
    or_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found the course—can I still enroll?', 'Is there still time to sign up for the LLM Zoomcamp?', 'Do I need to submit a project to get a certificate if I join late?', 'Can I start the program now and still earn a certificate?', 'Will I be able to get a certificate if I start after the enrollment deadline?']


In [48]:
usage.input_tokens, usage.output_tokens

(184, 419)

In [49]:
from Code.evaluation_utils import calc_price
cost = calc_price(usage)

cost

{'input_cost': 9.2e-06,
 'output_cost': 8.38e-05,
 'total_cost': 9.300000000000001e-05}

In [50]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found the course—can I still enroll?',
  'document': '74eb249bbf'},
 {'question': 'Is there still time to sign up for the LLM Zoomcamp?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit a project to get a certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Can I start the program now and still earn a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Will I be able to get a certificate if I start after the enrollment deadline?',
  'document': '74eb249bbf'}]

In [51]:
from Code.evaluation_utils import llm_structured_retry

In [52]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        or_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [53]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [54]:
from concurrent.futures import ThreadPoolExecutor
from Code.evaluation_utils import map_progress

In [55]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [56]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

517

In [57]:
from Code.evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.013150750000000001

In [58]:
from Code.evaluation_utils import calc_total_price

calc_total_price(usages)

0.013150750000000001

In [59]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [60]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)